# Smart Electricity Consumption Analytics
Run the cells in order. First upload your CSV files to the Databricks Volume and inspect their schemas.


## 00 Config And Inspection
Source: `databricks/00_config_and_inspection.py`


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "energy_analytics"
VOLUME = "energy_raw"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

ENERGY_FILE = f"{BASE_PATH}/corrected_daily_energy_met_MU.csv"
PEAK_FILE = f"{BASE_PATH}/corrected_peak_met_MW.csv"
SHORTAGE_FILE = f"{BASE_PATH}/peak_unmet_MW.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

def read_source(path):
    return (spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("mode", "PERMISSIVE")
            .csv(path))

energy_raw = read_source(ENERGY_FILE)
peak_raw = read_source(PEAK_FILE)
shortage_raw = read_source(SHORTAGE_FILE)

print("ENERGY SOURCE")
energy_raw.printSchema()
display(energy_raw.limit(10))

print("PEAK SOURCE")
peak_raw.printSchema()
display(peak_raw.limit(10))

print("SHORTAGE SOURCE")
shortage_raw.printSchema()
display(shortage_raw.limit(10))

print("Rows:", energy_raw.count(), peak_raw.count(), shortage_raw.count())


## 01 Bronze Ingestion
Source: `databricks/01_bronze_ingestion.py`


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "energy_analytics"
BASE = f"{CATALOG}.{SCHEMA}"

# If running in a fresh notebook, run/import notebook 00 first.
# The three DataFrames should be named energy_raw, peak_raw, shortage_raw.

def write_bronze(df, table_name):
    output = df.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_system", F.lit("India electricity CSV dataset"))
    (output.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BASE}.{table_name}"))
    print(f"Saved {BASE}.{table_name}: {output.count()} rows")

write_bronze(energy_raw, "bronze_energy")
write_bronze(peak_raw, "bronze_peak")
write_bronze(shortage_raw, "bronze_shortage")

display(spark.table(f"{BASE}.bronze_energy").limit(10))


## 02 Silver Transform
Source: `databricks/02_silver_transform.py`


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "energy_analytics"
BASE = f"{CATALOG}.{SCHEMA}"

# EDIT THESE VALUES after inspecting the source schemas.
# The default values match the included synthetic sample CSVs.
COLUMN_MAP = {
    "energy": {
        "date": "Date",
        "state": "State",
        "value": "Energy_Met_MU"
    },
    "peak": {
        "date": "Date",
        "state": "State",
        "value": "Peak_Met_MW"
    },
    "shortage": {
        "date": "Date",
        "state": "State",
        "value": "Peak_Unmet_MW"
    }
}

def standardize_source(df, date_col, state_col, value_col, metric_name):
    required = [date_col, state_col, value_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"{metric_name}: columns not found: {missing}. "
            f"Available columns: {df.columns}. Update COLUMN_MAP."
        )

    result = df.select(
        F.to_date(F.col(date_col)).alias("date"),
        F.initcap(F.trim(F.col(state_col).cast("string"))).alias("state"),
        F.regexp_replace(
            F.trim(F.col(value_col).cast("string")), ",", ""
        ).cast("double").alias(metric_name)
    )
    return result.filter(
        F.col("date").isNotNull() &
        F.col("state").isNotNull() &
        (F.col("state") != "")
    )

energy_silver = standardize_source(
    energy_raw,
    COLUMN_MAP["energy"]["date"],
    COLUMN_MAP["energy"]["state"],
    COLUMN_MAP["energy"]["value"],
    "energy_met_mu"
)

peak_silver = standardize_source(
    peak_raw,
    COLUMN_MAP["peak"]["date"],
    COLUMN_MAP["peak"]["state"],
    COLUMN_MAP["peak"]["value"],
    "peak_met_mw"
)

shortage_silver = standardize_source(
    shortage_raw,
    COLUMN_MAP["shortage"]["date"],
    COLUMN_MAP["shortage"]["state"],
    COLUMN_MAP["shortage"]["value"],
    "peak_unmet_mw"
)

# Remove duplicate keys within each source. If duplicates have a business
# meaning, aggregate them instead of arbitrarily selecting a record.
energy_silver = energy_silver.dropDuplicates(["date", "state"])
peak_silver = peak_silver.dropDuplicates(["date", "state"])
shortage_silver = shortage_silver.dropDuplicates(["date", "state"])

# Full outer joins preserve records that occur in only one source.
silver = (
    energy_silver
    .join(peak_silver, ["date", "state"], "full")
    .join(shortage_silver, ["date", "state"], "full")
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("region", F.lit("Unknown"))
    .withColumn("_processed_at", F.current_timestamp())
)

# Do not replace missing energy/demand values with zero: null may mean
# unavailable data, not zero consumption.
(silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BASE}.silver_electricity"))

print("Silver table created:", silver.count(), "rows")
display(silver.orderBy("date", "state").limit(20))

# Data quality summary
display(silver.select(
    F.count("*").alias("row_count"),
    F.countDistinct("state").alias("state_count"),
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.sum(F.col("energy_met_mu").isNull().cast("int")).alias("missing_energy"),
    F.sum(F.col("peak_met_mw").isNull().cast("int")).alias("missing_peak"),
    F.sum(F.col("peak_unmet_mw").isNull().cast("int")).alias("missing_shortage")
))


## 03 Gold Analytics
Source: `databricks/03_gold_analytics.py`


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "energy_analytics"
BASE = f"{CATALOG}.{SCHEMA}"

silver = spark.table(f"{BASE}.silver_electricity")

# 1. State-level KPIs
gold_state = (
    silver.groupBy("state")
    .agg(
        F.sum("energy_met_mu").alias("total_energy_met_mu"),
        F.avg("energy_met_mu").alias("avg_daily_energy_met_mu"),
        F.max("energy_met_mu").alias("max_daily_energy_met_mu"),
        F.avg("peak_met_mw").alias("avg_peak_met_mw"),
        F.max("peak_met_mw").alias("max_peak_met_mw"),
        F.sum("peak_unmet_mw").alias("total_peak_unmet_mw"),
        F.countDistinct("date").alias("days_recorded")
    )
)
gold_state.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_state_consumption")

# 2. Monthly trends
gold_monthly = (
    silver.groupBy("year", "month", "month_name")
    .agg(
        F.sum("energy_met_mu").alias("total_energy_met_mu"),
        F.avg("energy_met_mu").alias("avg_daily_energy_met_mu"),
        F.max("peak_met_mw").alias("max_peak_met_mw"),
        F.avg("peak_unmet_mw").alias("avg_daily_peak_unmet_mw")
    )
)
gold_monthly.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_monthly_trends")

# 3. Daily peak demand by state
gold_peak = (
    silver.groupBy("date", "year", "month", "state")
    .agg(
        F.max("peak_met_mw").alias("peak_met_mw"),
        F.max("peak_unmet_mw").alias("peak_unmet_mw")
    )
)
gold_peak.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_peak_demand")

# 4. Shortage analysis
gold_shortage = (
    silver.groupBy("state", "year", "month")
    .agg(
        F.sum("peak_unmet_mw").alias("total_peak_unmet_mw"),
        F.avg("peak_unmet_mw").alias("avg_peak_unmet_mw"),
        F.max("peak_unmet_mw").alias("max_peak_unmet_mw"),
        F.count(F.when(F.col("peak_unmet_mw") > 0, True)).alias("shortage_days")
    )
)
gold_shortage.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_power_shortage")

for table in [
    "gold_state_consumption",
    "gold_monthly_trends",
    "gold_peak_demand",
    "gold_power_shortage"
]:
    print(table)
    display(spark.table(f"{BASE}.{table}").limit(20))


## 04 Validation And Export
Source: `databricks/04_validation_and_export.py`


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "energy_analytics"
BASE = f"{CATALOG}.{SCHEMA}"
EXPORT_PATH = "/Volumes/workspace/energy_analytics/energy_raw/powerbi_exports"

gold_tables = [
    "gold_state_consumption",
    "gold_monthly_trends",
    "gold_peak_demand",
    "gold_power_shortage"
]

for table_name in gold_tables:
    df = spark.table(f"{BASE}.{table_name}")
    print(f"{table_name}: {df.count()} rows")
    if df.count() == 0:
        raise ValueError(f"{table_name} is empty")
    display(df.limit(5))

# Export each table as a folder containing part-*.csv.
# In Databricks, download the generated CSV part file(s) for Power BI.
for table_name in gold_tables:
    output_path = f"{EXPORT_PATH}/{table_name}"
    (spark.table(f"{BASE}.{table_name}")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .csv(output_path))
    print("Exported:", output_path)

print("Validation and export completed.")
